# register-buffer — ex2: two-state Module: one buffer + one Parameter — show buffer is in state_dict but NOT in .parameters()

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `register-buffer`. Running the final beacon cell reports progress against the `PyTorch: register_buffer` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: register_buffer` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`register-buffer`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "register-buffer"
DD_SUBTOPIC = "PyTorch: register_buffer"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `register_buffer` vs `nn.Parameter` — state_dict yes, parameters no

Ex1 registered non-trainable state via `register_buffer` and verified it appears in `state_dict()`. The deepening move pins the CONTRAST against `nn.Parameter`:

| trait                              | `register_buffer` | `nn.Parameter` |
|------------------------------------|-------------------|----------------|
| appears in `state_dict()`          | yes               | yes            |
| appears in `.parameters()`         | NO                | yes            |
| receives gradients                 | NO                | yes (if `requires_grad`)|
| moves with `.to(device)`           | yes               | yes            |
| saved on `torch.save(model)`       | yes               | yes            |

**Why both are needed.** BN's `running_mean` MUST be saved (state_dict) AND must travel with the model to GPU (`.to(device)`) — but it MUST NOT be optimised by `torch.optim` (no gradient). Buffers exist exactly for this 'persistent but non-trainable' slot.

**The diagnostic.** `len(list(module.parameters()))` should be 0 for a pure-buffer module; `len(module.state_dict())` should equal the number of registered buffers.

### Exercise 2 — two-state Module: one buffer + one Parameter — show buffer is in state_dict but NOT in .parameters()

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyse the membership contract of `register_buffer` vs `nn.Parameter` by building a Module with ONE of each, then verifying: both appear in `state_dict()`; only the Parameter appears in `.parameters()`; neither requires_grad propagates to the buffer.
> Keywords: register_buffer, nn.Parameter, state_dict, parameters
> ```

**KCs targeted:** `buffer-in-state-dict-not-in-parameters`, `parameter-in-both-state-dict-and-parameters`

Implement `ex2_MixedStateModule` as a subclass of `nn.Module`:

Constructor `__init__(self, dim)`:
1. `super().__init__()`.
2. Register a NON-TRAINABLE buffer named `'fixed_scale'`, initialised to `torch.ones(dim)` — use `self.register_buffer(...)`.
3. Register a TRAINABLE parameter named `'weight'`, initialised to `torch.zeros(dim)` — use `nn.Parameter(...)` and assignment.

Forward `forward(self, x)`:
- Return `self.fixed_scale * x + self.weight`. (A simple affine for the test to exercise; nothing tricky.)

The test will verify state_dict / parameters membership and behaviour under `.to(device)` and `requires_grad`.

Inputs to `forward`: `x` of shape `(..., dim)`. Output: same shape.

In [ ]:
import torch as _t
import torch.nn as nn

class ex2_MixedStateModule(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.register_buffer('fixed_scale', _t.ones(dim))
        self.weight = nn.Parameter(_t.zeros(dim))

    def forward(self, x):
        return self.fixed_scale * x + self.weight


<details><summary>Solution</summary>

```python
import torch as _t
import torch.nn as nn

class ex2_MixedStateModule(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.register_buffer('fixed_scale', _t.ones(dim))
        self.weight = nn.Parameter(_t.zeros(dim))

    def forward(self, x):
        return self.fixed_scale * x + self.weight
```

**`register_buffer(name, tensor)` over `self.name = tensor`.** A plain attribute assignment puts the tensor on the module but OUT of state_dict — it won't save/load, won't move with `.to(device)`. `register_buffer` is the documented hook.

**`nn.Parameter(tensor)` automatically registers in both state_dict and .parameters().** No `register_parameter` call needed when you assign via `self.weight = nn.Parameter(...)` — `nn.Module.__setattr__` notices the type and wires it up.

**The optimizer-step test is the load-bearing one.** It's not just about API surface (`'in state_dict'` vs `'in parameters'`); it's about RUNTIME effect: passing `model.parameters()` to `SGD(...)` automatically excludes buffers from updates. This is why BN's running_mean stays fixed under SGD even though it appears in `state_dict`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()